In [1]:
import os
import torch
from PIL import Image
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, EulerAncestralDiscreteScheduler


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device.upper()}")
print(f"CUDA available: {torch.cuda.is_available()}")

output_text2img_dir = "./output_text2img"
output_img2img_dir = "./output_img2img"
os.makedirs(output_text2img_dir, exist_ok=True)
os.makedirs(output_img2img_dir, exist_ok=True)

model_id = "stabilityai/stable-diffusion-2-1-base"

print("\n--- Cargando modelo para Text-to-Image ---")
pipe_txt = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe_txt.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe_txt.scheduler.config)
pipe_txt.to(device)

prompt_txt = "A high-resolution product photo of modern running sneakers on a white background, studio lighting, highly detailed"

print(f"Generando imagen desde el prompt: '{prompt_txt}'...")
image_generated = pipe_txt(
    prompt=prompt_txt,
    num_inference_steps=30,  
    guidance_scale=7.5      
).images[0]


txt_output_path = os.path.join(output_text2img_dir, "sneaker_generated.png")
image_generated.save(txt_output_path)
print(f"Imagen guardada con éxito en: {txt_output_path}")

del pipe_txt
if device == "cuda":
    torch.cuda.empty_cache()


print("\n--- Cargando modelo para Image-to-Image ---")

pipe_img = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe_img.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe_img.scheduler.config)
pipe_img.to(device)

init_image = Image.open(txt_output_path).convert("RGB")
init_image = init_image.resize((512, 512)) 


prompt_img2img = "A futuristic version of the product with neon accents and a dark background, cyberpunk style, 8k resolution"

print(f"Generando variación desde la imagen base con el prompt: '{prompt_img2img}'...")

image_variation = pipe_img(
    prompt=prompt_img2img,
    image=init_image,
    strength=0.75,           
    num_inference_steps=30,
    guidance_scale=8.0
).images[0]

#
img_output_path = os.path.join(output_img2img_dir, "sneaker_futuristic_variation.png")
image_variation.save(img_output_path)
print(f"Variación guardada con éxito en: {img_output_path}")

/opt/anaconda3/lib/python3.13/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6ab95625-4866ac8600dedd384b183232;72260e31-6f77-46e6-ac46-a2cb40b19609)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1-base.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permi

Dispositivo detectado: CPU
CUDA available: False

--- Cargando modelo para Text-to-Image ---


OSError: Cannot load model stabilityai/stable-diffusion-2-1-base: the model is not fully cached locally (Cannot find an appropriate cached snapshot folder for the specified revision on the local disk and outgoing traffic has been disabled. To enable repo look-ups and downloads online, pass 'local_files_only=False' as input.) and an error occurred while trying to fetch metadata from the Hub. Please check out the root cause in the stacktrace above.

In [ ]:
nombre_imagen = "tenis_base.png"  # <- Cambia esto por el nombre real de tu archivo
ruta_imagen_base = os.path.join("./custom_dataset", nombre_imagen)

print(f"Cargando imagen base desde: {ruta_imagen_base}")
init_image = Image.open(ruta_imagen_base).convert("RGB")
init_image = init_image.resize((512, 512))
# --------------------------------------------------

prompt_img2img = "A futuristic version of the product."
print(f"Generando variación desde la imagen base con el prompt: '{prompt_img2img}'...")

image_variation = pipe_img(
    prompt=prompt_img2img, 
    image=init_image, 
    strength=0.75, 
    num_inference_steps=30, 
    guidance_scale=8.0
).images[0]

img_output_path = os.path.join(output_img2img_dir, "sneaker_futuristic_variation.png")
image_variation.save(img_output_path)
print(f"Variación guardada con éxito en: {img_output_path}")